# Trading and risk

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

This notebook is a software tutorial using bundled example bars or a small deterministic fixture. It is not a trading recommendation or a research performance claim.

A trade plan fixes entry, stop, and target before computing size. We use the production sizing functions to show how dollar risk depends on stop distance and lot size.

In [ ]:
import matplotlib.pyplot as plt

from trading.risk.sizing import compute_lots, compute_sl_tp_long, stop_loss_risk_usd

entry = 100.0
stop, target = compute_sl_tp_long(entry, last_swing_low=96.0, buffer_pts=0.5, rr_ratio=2.0)
requested_risk_usd = 200.0
lots = compute_lots(
    equity=10_000.0,
    risk_frac=0.02,
    entry_price=entry,
    sl_price=stop,
    min_lot=0.01,
    max_lot=100.0,
    risk_usd=requested_risk_usd,
)
actual_risk = stop_loss_risk_usd(entry - stop, lots, contract_size=1.0)
print(
    {
        "entry": entry,
        "stop": stop,
        "target": target,
        "lots": round(lots, 3),
        "risk_usd": round(actual_risk, 2),
        "reward_risk": (target - entry) / (entry - stop),
    }
)

Figure: long entry, stop, and target levels on a price axis; green marks the target, blue the entry, and red the stop.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hlines([3, 2, 1], xmin=96, xmax=target, colors=["#2a8c57", "#315f9b", "#b84a4a"], linewidth=6)
ax.text(entry, 3.12, f"Entry {entry:g}", ha="center")
ax.text(stop, 1.68, f"Stop {stop:g}", ha="center")
ax.text(target, 2.12, f"Target {target:g}", ha="center")
ax.set(
    xlim=(95, target + 1),
    ylim=(0.5, 3.6),
    yticks=[],
    xlabel="Price points",
    title="Long trade geometry (not to scale)",
)
ax.grid(axis="x", alpha=0.2)
plt.show()
assert round(actual_risk, 2) == requested_risk_usd

## Interpret the output

The stop is 4.5 points from entry, so the risk budget determines position size. Spread, fill behavior, and account guardrails can change realized results; a planned risk amount is not a guaranteed loss ceiling in every market condition.

See [`compute_lots`](../../api/domains/trading), [risk and account limits](../../concepts/risk_and_account_limits), and [risk/execution reference](../../reference/risk_and_execution).